In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/projects/immunestatus/vdjdb/pools/human.tra.aa.txt', sep='\t')

In [3]:
df

,count,freq,cdr3nt,cdr3aa,v,d,j,VEnd,DStart,DEnd,JStart,incidence,convergence,occurrences
0,535708,7.055747e-03,TGTGCTGTGATGGATAGCAACTATCAGTTAATCTGG,CAVMDSNYQLIW,TRAV1-2,.,TRAJ33,0,-1,-1,10,1,47,47
1,435850,5.740529e-03,TGTGCTGTGAGAGATAGCAACTATCAGTTAATCTGG,CAVRDSNYQLIW,TRAV1-2,.,TRAJ33,0,-1,-1,12,1,90,90
2,229349,3.020729e-03,TGTGCTTATAGGAGCGCGAACTTCAACAAATTTTACTTT,CAYRSANFNKFYF,TRAV38-2/DV8,TRDD1,TRAJ21,0,7,10,18,1,6,6
3,211244,2.782270e-03,TGTGCTGTGCTGGATAGCAACTATCAGTTAATCTGG,CAVLDSNYQLIW,TRAV1-2,.,TRAJ33,0,-1,-1,10,1,102,102
4,208416,2.745023e-03,TGTGCTGTCCTTTGCTCTGGCAACACAGGCAAACTAATCTTT,CAVLCSGNTGKLIF,TRAV21,.,TRAJ37,0,-1,-1,13,1,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266269,1,1.317088e-08,TGTGCCGTGAACTCCCCAGGAGGAGGAGGAAACAAACTCACCTTT,CAVNSPGGGGNKLTF,TRAV12-2,.,TRAJ10,11,-1,-1,21,1,1,1
2266270,1,1.317088e-08,TGTGCTGCGAACGGTCCCGTTGGAAATGAGAAATTAACCTTT,CAANGPVGNEKLTF,TRAV21,.,TRAJ48,9,-1,-1,19,1,1,1
2266271,1,1.317088e-08,TGTGCTGTGCAGGCACTAGCTGACAGCTGGGGGAAATTCCAGTTT,CAVQALADSWGKFQF,TRAV20,.,TRAJ24,13,-1,-1,19,1,1,1
2266272,1,1.317088e-08,TGTGCAGCAGGACCTCGAGGTAGCAACTATAAACTGACATTT,CAAGPRGSNYKLTF,TRAV23/DV6,TRDD2,TRAJ53,0,12,14,16,1,1,1


In [4]:
number_of_reads_overall = df['count'].sum()

In [5]:
import numpy as np
import pandas as pd

# df: большие данные с колонкой 'count' и любыми другими (cdr3nt, cdr3aa, ...)
counts = df['count'].to_numpy(dtype=np.int64)
target = 5_000_000
total = counts.sum()
p = target / total

# Биномиальное прореживание
keep = np.random.binomial(counts, p)

# Оставляем только строки с ненулевым отбором и «разворачиваем»
mask = keep > 0
sampled = df.loc[mask].copy()
sampled_counts = keep[mask]

# Если нужны именно риды как строки (а не агрегированные счётчики):
# индексы исходных строк, повторённые по числу отобранных ридов
idx_rep = np.repeat(sampled.index.to_numpy(), sampled_counts)
sampled_reads = df.loc[idx_rep]            # это 1e6 строк (примерно)


In [6]:
sampled = sampled_reads[['cdr3aa', 'v', 'j']].drop_duplicates()
sampled

,cdr3aa,v,j
0,CAVMDSNYQLIW,TRAV1-2,TRAJ33
1,CAVRDSNYQLIW,TRAV1-2,TRAJ33
2,CAYRSANFNKFYF,TRAV38-2/DV8,TRAJ21
3,CAVLDSNYQLIW,TRAV1-2,TRAJ33
4,CAVLCSGNTGKLIF,TRAV21,TRAJ37
...,...,...,...
2266196,CALTSESGGSNYKLTF,TRAV9-2,TRAJ53
2266218,CVVSPFNAGGTSYGKLTF,TRAV8-2,TRAJ52
2266231,CAVPAPSGATNQFYF,TRAV20,TRAJ49
2266241,CAQFFGNEKLTF,TRAV38-2/DV8,TRAJ48


In [7]:
sampled['locus'] = 'alpha'

In [8]:
sampled = sampled.rename(columns={'cdr3aa': 'junction_aa', 
                        'v': 'v_call',
                        'j': 'j_call'})
sampled

,junction_aa,v_call,j_call,locus
0,CAVMDSNYQLIW,TRAV1-2,TRAJ33,alpha
1,CAVRDSNYQLIW,TRAV1-2,TRAJ33,alpha
2,CAYRSANFNKFYF,TRAV38-2/DV8,TRAJ21,alpha
3,CAVLDSNYQLIW,TRAV1-2,TRAJ33,alpha
4,CAVLCSGNTGKLIF,TRAV21,TRAJ37,alpha
...,...,...,...,...
2266196,CALTSESGGSNYKLTF,TRAV9-2,TRAJ53,alpha
2266218,CVVSPFNAGGTSYGKLTF,TRAV8-2,TRAJ52,alpha
2266231,CAVPAPSGATNQFYF,TRAV20,TRAJ49,alpha
2266241,CAQFFGNEKLTF,TRAV38-2/DV8,TRAJ48,alpha


In [9]:
sampled.to_csv('/projects/immunestatus/vdjdb/airr_format/tra_background.tsv', sep='\t', index=False)

In [19]:
df = pd.read_csv('/projects/immunestatus/vdjdb/vdjdb-2025-07-30/vdjdb_full_filtered.txt', sep='\t')

/scratch/ipykernel_405796/3455155727.py:1: DtypeWarning: Columns (5,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/projects/immunestatus/vdjdb/vdjdb-2025-07-30/vdjdb_full_filtered.txt', sep='\t')


In [20]:
df = df[df.species == 'HomoSapiens']

In [21]:
df[df['cdr3.alpha'].notna()][['cdr3.alpha', 'v.alpha', 'j.alpha', 'antigen.epitope']]

,cdr3.alpha,v.alpha,j.alpha,antigen.epitope
163,CAASGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF
164,CAVSGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF
165,CAESGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF
166,CAAYGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF
167,CAVSGTYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF
...,...,...,...,...
107922,CAFDNQAGTALIF,TRAV24*01,TRAJ15*01,FRDYVDRFYKTLRAEQASQE
107923,CASYGGATNKLIF,TRAV24*01,TRAJ32*01,FRDYVDRFYKTLRAEQASQE
107924,CAYCGGSPNNLIF,TRAV24*01,TRAJ42*01,FRDYVDRFYKTLRAEQASQE
107925,CASDYGGSQGNLIF,TRAV24*01,TRAJ42*01,FRDYVDRFYKTLRAEQASQE


In [22]:
common_epitopes = df[df['cdr3.alpha'].notna()][
    ['cdr3.alpha', 'v.alpha', 'j.alpha', 'antigen.epitope']]['antigen.epitope'].value_counts().head(10).index

In [23]:
df = df[df['cdr3.alpha'].notna()][
    ['cdr3.alpha', 'v.alpha', 'j.alpha', 'antigen.epitope']].dropna().drop_duplicates().reset_index(names='clone_id')
df['locus'] = 'alpha'

In [24]:
df = df.rename(columns={'cdr3.alpha': 'junction_aa', 'v.alpha': 'v_call', 'j.alpha': 'j_call' })

In [25]:
df

,clone_id,junction_aa,v_call,j_call,antigen.epitope,locus
0,163,CAASGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF,alpha
1,164,CAVSGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF,alpha
2,165,CAESGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF,alpha
3,166,CAAYGGYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF,alpha
4,167,CAVSGTYQKVTF,TRAV5*01,TRAJ13*01,KAFSPEVIPMF,alpha
...,...,...,...,...,...,...
33881,107919,CVFQTGGNNLFF,TRAV24*01,TRAJ36*01,FRDYVDRFYKTLRAEQASQE,alpha
33882,107920,CACYGGATNKLIF,TRAV24*01,TRAJ32*01,FRDYVDRFYKTLRAEQASQE,alpha
33883,107921,CAFDNQAANNLIF,TRAV24*01,TRAJ15*01,FRDYVDRFYKTLRAEQASQE,alpha
33884,107924,CAYCGGSPNNLIF,TRAV24*01,TRAJ42*01,FRDYVDRFYKTLRAEQASQE,alpha


In [26]:
for epi in common_epitopes:
    vdjdb_epi = df[df['antigen.epitope'] == epi].drop(columns=['antigen.epitope'])
    vdjdb_epi.to_csv(
        f'/projects/immunestatus/vdjdb/airr_format/tra_vdjdb_uni_{epi}.tsv', sep='\t', index=False)

In [27]:
df.to_csv(
    '/projects/immunestatus/vdjdb/airr_format/tra_vdjdb_uni.tsv', sep='\t', index=False)